In [ ]:
# day 4 imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd

DATA_PATH = PROJECT_ROOT / "data" / "cleaned" / "adult_cleaned.csv"
FIGURES_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR = PROJECT_ROOT / "results"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, dtype={"zip_code": "string"})
df = df.dropna(subset=["income"]).copy()

df["income"] = df["income"].astype(str).str.strip()

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (32561, 16)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,zip_code
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K,93810
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K,24592
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K,13278
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K,46048
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K,42098


In [8]:
# Load your cleaned dataset

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd

DATA_PATH = PROJECT_ROOT / "data" / "cleaned" / "adult_cleaned.csv"

df = pd.read_csv(DATA_PATH, dtype={"zip_code": "string"})
df = df.dropna(subset=["income"]).copy()
df["income"] = df["income"].astype(str).str.strip()

print("Dataset shape:", df.shape)
display(df.head())


Dataset shape: (32561, 16)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,zip_code
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K,93810
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K,24592
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K,13278
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K,46048
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K,42098


In [9]:
# Create the train/test split

X = df.drop(columns=["income"])
y = df["income"]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 26048
Testing records: 6513


In [10]:
# Create the preprocessing pipeline

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


In [11]:
# Train both models and generate predictions

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

trained_pipelines = {}
predictions = {}

for model_name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    pipeline.fit(X_train, y_train)

    predictions[model_name] = pipeline.predict(X_test)
    trained_pipelines[model_name] = pipeline

    print(f"{model_name} trained successfully.")

print("Available models:", list(predictions.keys()))

Logistic Regression trained successfully.
Random Forest trained successfully.
Available models: ['Logistic Regression', 'Random Forest']


In [12]:
#Import the utility

import sys
from pathlib import Path

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd

sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import (
    evaluate_model,
    append_result,
    print_evaluation_report,
)

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_LOG = RESULTS_DIR / "results_log.csv"

print("Results log:", RESULTS_LOG)

Results log: d:\VS Code\Projects\python\Internship\results\results_log.csv


In [13]:
# Load the baseline predictions

print("Available models:", list(predictions.keys()))
print("Test records:", len(y_test))

assert len(y_test) > 0

for model_name, y_pred in predictions.items():
    assert len(y_pred) == len(y_test)

Available models: ['Logistic Regression', 'Random Forest']
Test records: 6513


In [14]:
# Evaluate and log both models

for model_name, y_pred in predictions.items():
    result = evaluate_model(
        y_true=y_test,
        y_pred=y_pred,
        model_name=model_name,
        experiment="baseline",
        anonymization_level="none",
        positive_label=">50K",
    )

    append_result(result, RESULTS_LOG)

    print(f"\n{model_name}")
    print(pd.Series(result).to_string())


Logistic Regression
experiment                        baseline
anonymization_level                   none
model                  Logistic Regression
accuracy                          0.855059
precision                          0.74037
recall                            0.612883
f1_score                          0.670621

Random Forest
experiment                  baseline
anonymization_level             none
model                  Random Forest
accuracy                     0.86074
precision                    0.77068
recall                      0.600128
f1_score                    0.674794


In [15]:
# Display the cumulative results log

results_df = pd.read_csv(RESULTS_LOG)

display(results_df.round(4))

print("Total logged experiments:", len(results_df))
print("Saved to:", RESULTS_LOG)

,experiment,anonymization_level,model,accuracy,precision,recall,f1_score
0,baseline,none,Logistic Regression,0.8551,0.7404,0.6129,0.6706
1,baseline,none,Random Forest,0.8607,0.7707,0.6001,0.6748


Total logged experiments: 2
Saved to: d:\VS Code\Projects\python\Internship\results\results_log.csv


In [16]:
# Generate a detailed report


for model_name, y_pred in predictions.items():
    print(f"\n{'=' * 50}")
    print(model_name)
    print("=" * 50)

    print_evaluation_report(y_test, y_pred)


Logistic Regression
Classification Report:
              precision    recall  f1-score   support

       <=50K       0.88      0.93      0.91      4945
        >50K       0.74      0.61      0.67      1568

    accuracy                           0.86      6513
   macro avg       0.81      0.77      0.79      6513
weighted avg       0.85      0.86      0.85      6513

Confusion Matrix:
[[4608  337]
 [ 607  961]]

Random Forest
Classification Report:
              precision    recall  f1-score   support

       <=50K       0.88      0.94      0.91      4945
        >50K       0.77      0.60      0.67      1568

    accuracy                           0.86      6513
   macro avg       0.83      0.77      0.79      6513
weighted avg       0.85      0.86      0.85      6513

Confusion Matrix:
[[4665  280]
 [ 627  941]]
